# Merit Order Curve Construction — Fjord Energy Portfolio (NO1-NO5)

**Scenario:** Fjord Energy AS is a Nordic portfolio aggregator managing a mixed fleet of
generation assets across the NO1-NO5 bidding zones:

| Asset | Zone | Type | Capacity (MW) |
|-------|------|------|---------------|
| Hallingdal Wind | NO2 | Wind (variable) | 120 |
| Voss Hydro | NO5 | Run-of-river hydro | 200 |
| Rjukan Hydro | NO1 | Reservoir hydro | 300 |
| Sola Gas | NO2 | Open-cycle gas turbine | 150 |

For each MTU the trading desk aggregates individual asset supply curves into a
**portfolio merit order curve** — the foundation for understanding portfolio clearing price
and revenue attribution.

This notebook demonstrates:
- Building multi-step supply curves from asset cost data using `from_dict_list`
- Using `merge_curves` to aggregate across assets for the same MTU
- Transforming curves (`scale_curve`, `clip_curve`, price shifting)
- Visualising individual and aggregate merit order curves

## Prerequisites

```bash
pip install nexa-bidkit matplotlib
# or, from source:
poetry install
```

In [ ]:
from datetime import datetime, timedelta
from decimal import Decimal
import math
import zoneinfo

import matplotlib.pyplot as plt
import pandas as pd

from nexa_bidkit.curves import (
    from_dict_list,
    merge_curves,
    scale_curve,
    clip_curve,
    to_dataframe as curves_to_df,
    get_curve_summary,
)
from nexa_bidkit.types import (
    CurveType,
    MTUDuration,
    MTUInterval,
    PriceQuantityCurve,
    PriceQuantityStep,
)

CET = zoneinfo.ZoneInfo("Europe/Oslo")
print("nexa-bidkit imports OK")

## 1. Define a Reference MTU

We'll build curves for a single MTU (09:15–09:30 CET on 2026-03-15) to illustrate
the aggregation, then extend to the full day profile.

In [ ]:
delivery_day = datetime(2026, 3, 15, tzinfo=CET)
MTU = MTUDuration.QUARTER_HOURLY

# Reference MTU: 09:15–09:30 CET (high-demand morning period)
ref_mtu = MTUInterval.from_start(
    start=delivery_day + timedelta(hours=9, minutes=15),
    duration=MTU,
)
print(f"Reference MTU: {ref_mtu.start.astimezone(CET)} → {ref_mtu.end.astimezone(CET)}")

## 2. Individual Asset Supply Curves

Each asset contributes a supply curve reflecting its cost structure:

- **Wind**: Zero-marginal-cost renewable; low price floors, high uncertainty at top volumes
- **Run-of-river hydro**: Low marginal cost, limited dispatchability (flow constraints)
- **Reservoir hydro**: Flexible dispatch; opportunity cost of stored water sets prices
- **Gas turbine**: High marginal cost (fuel + CO₂); only economic at high prices

We use `from_dict_list` to construct curves from plain Python dictionaries.

In [ ]:
def supply_curve(steps_data: list[tuple[float, float]], mtu: MTUInterval) -> PriceQuantityCurve:
    """Build a supply curve from (price_eur, volume_mw) tuples via from_dict_list."""
    return from_dict_list(
        steps=[{"price": p, "volume": v} for p, v in steps_data],
        curve_type=CurveType.SUPPLY,
        mtu=mtu,
    )

# Hallingdal Wind: 120 MW capacity, morning wind at ~80 MW
wind_curve = supply_curve([
    (5.0, 32.0),   # firm 40%: almost free (renewable incentive)
    (20.0, 32.0),  # mid 40%: low cost
    (45.0, 16.0),  # top 20%: forecast uncertainty premium
], ref_mtu)

# Voss Run-of-River Hydro: 200 MW, near-full output in spring
ror_hydro_curve = supply_curve([
    (2.0, 80.0),   # base flow: nearly zero marginal cost
    (8.0, 80.0),   # higher flow band
    (15.0, 40.0),  # regulated flow (slightly restricted)
], ref_mtu)

# Rjukan Reservoir Hydro: 300 MW, flexible with opportunity cost
reservoir_hydro_curve = supply_curve([
    (25.0, 60.0),   # base dispatch: water already scheduled
    (40.0, 120.0),  # mid dispatch: moderate opportunity cost
    (60.0, 80.0),   # peak dispatch: high opportunity cost (saving for evening)
    (80.0, 40.0),   # emergency dispatch: very high opportunity cost
], ref_mtu)

# Sola Gas Turbine: 150 MW, high marginal cost (fuel + CO2)
# Gas price EUR 35/MWh, efficiency 38%, CO2 EUR 75/tonne at 0.18 tCO2/MWh
gas_fuel_cost = 35.0 / 0.38          # ~92 EUR/MWh
gas_co2_cost = 75.0 * 0.18           # ~13.5 EUR/MWh
gas_vom = 4.0                         # variable O&M
gas_marginal = gas_fuel_cost + gas_co2_cost + gas_vom  # ~109.5 EUR/MWh

gas_curve = supply_curve([
    (gas_marginal, 100.0),           # base dispatch
    (gas_marginal + 10.0, 30.0),     # contingency (less efficient)
    (gas_marginal + 25.0, 20.0),     # emergency capacity
], ref_mtu)

print("Asset curves for MTU 09:15–09:30 CET:")
for name, curve in [
    ("Wind", wind_curve), ("RoR Hydro", ror_hydro_curve),
    ("Reservoir Hydro", reservoir_hydro_curve), ("Gas", gas_curve)
]:
    s = get_curve_summary(curve)
    print(f"  {name:<20}: {float(s['total_volume']):6.0f} MW, "
          f"price range {float(s['min_price']):.1f}–{float(s['max_price']):.1f} EUR/MWh")

## 3. Aggregate Portfolio Merit Order Curve

`merge_curves` combines all asset curves for the same MTU into a single portfolio
supply curve by aggregating volumes at each price level.

In [ ]:
portfolio_curve = merge_curves(
    [wind_curve, ror_hydro_curve, reservoir_hydro_curve, gas_curve],
    aggregation="sum",
)

print("Portfolio merit order curve (09:15–09:30 CET):")
df = curves_to_df(portfolio_curve)
df["price"] = df["price"].apply(float)
df["volume"] = df["volume"].apply(float)
df["cumulative_volume"] = df["volume"].cumsum()
print(df.to_string(index=False))

total = float(portfolio_curve.total_volume)
print(f"\nTotal portfolio volume: {total:.0f} MW")

## 4. Visualisation — Merit Order Curves

Plot individual asset curves overlaid on the aggregate portfolio curve.
The merit order shows the cheapest generation sources dispatched first.

In [ ]:
def curve_to_step_xy(curve: PriceQuantityCurve) -> tuple[list[float], list[float]]:
    """Convert a supply curve to step-function (cumulative volume, price) vectors."""
    x = [0.0]
    y_steps: list[float] = []
    for step in curve.steps:
        y_steps.extend([float(step.price), float(step.price)])
        x.extend([x[-1], x[-1] + float(step.volume)])
    return x[:-1], y_steps

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    "Fjord Energy Portfolio — Merit Order Curves (09:15–09:30 CET, 2026-03-15)",
    fontsize=13, fontweight="bold"
)

# ---- Left: individual assets ----
asset_info = [
    ("Hallingdal Wind (NO2)", wind_curve, "#27ae60"),
    ("Voss RoR Hydro (NO5)", ror_hydro_curve, "#2980b9"),
    ("Rjukan Reservoir (NO1)", reservoir_hydro_curve, "#8e44ad"),
    ("Sola Gas OCGT (NO2)", gas_curve, "#e74c3c"),
]

for name, curve, color in asset_info:
    x, y = curve_to_step_xy(curve)
    ax1.step(x, y, where="post", color=color, linewidth=2, label=name)
    ax1.fill_between(x, y, alpha=0.12, color=color, step="post")

ax1.set_title("Individual Asset Curves", fontsize=11)
ax1.set_xlabel("Cumulative Volume (MW)")
ax1.set_ylabel("Price (EUR/MWh)")
ax1.legend(fontsize=8, loc="upper left")
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0, 145)

# ---- Right: portfolio aggregate ----
x_p, y_p = curve_to_step_xy(portfolio_curve)
ax2.step(x_p, y_p, where="post", color="#2c3e50", linewidth=2.5, label="Portfolio")
ax2.fill_between(x_p, y_p, alpha=0.15, color="#2c3e50", step="post")

# Mark a hypothetical market clearing price
CLEARING_PRICE = 50.0
ax2.axhline(CLEARING_PRICE, color="#e74c3c", linestyle="--", linewidth=1.5,
            label=f"Clearing price: {CLEARING_PRICE} EUR/MWh")

# Find cleared volume at clearing price
cleared_vol = sum(
    float(s.volume) for s in portfolio_curve.steps if float(s.price) <= CLEARING_PRICE
)
ax2.axvline(cleared_vol, color="#e74c3c", linestyle=":", alpha=0.7)
ax2.text(cleared_vol + 5, 2, f"{cleared_vol:.0f} MW cleared",
         color="#e74c3c", fontsize=9)

ax2.set_title("Aggregate Portfolio Curve", fontsize=11)
ax2.set_xlabel("Cumulative Volume (MW)")
ax2.set_ylabel("Price (EUR/MWh)")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 145)

plt.tight_layout()
plt.savefig("merit_order_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved: merit_order_curves.png")

## 5. Intraday Profile — Volume Across the Day

Now we extend the analysis across all 96 MTUs, simulating how asset output and gas
marginal cost change through the delivery day.

In [ ]:
def gas_price_at_hour(hour: float) -> float:
    """Simulate intraday gas price variation (EUR/MWh)."""
    base = 35.0
    morning_peak = 3.0 * math.exp(-((hour - 8) ** 2) / 4)
    evening_peak = 2.5 * math.exp(-((hour - 18) ** 2) / 4)
    return base + morning_peak + evening_peak

# Wind forecast across the day
hourly_wind = [55, 58, 62, 70, 75, 78, 80, 82, 85, 88, 90, 88,
               85, 80, 75, 70, 68, 65, 60, 55, 52, 50, 48, 50]
wind_forecast = [mw for mw in hourly_wind for _ in range(4)]

# Build all 96 MTU intervals
mtu_list = [
    MTUInterval.from_start(delivery_day + timedelta(minutes=15 * i), MTU)
    for i in range(96)
]

vols_wind, vols_ror, vols_res, vols_gas, marginal_prices = [], [], [], [], []

for i, mtu in enumerate(mtu_list):
    hour = i / 4  # fractional hour

    w_mw = wind_forecast[i]
    wc = supply_curve([(5.0, w_mw * 0.4), (20.0, w_mw * 0.4), (45.0, w_mw * 0.2)], mtu)
    ror = supply_curve([(2.0, 80.0), (8.0, 80.0), (15.0, 40.0)], mtu)

    opp_cost = 25.0 + 10.0 * math.sin(math.pi * hour / 12)
    res = supply_curve([(opp_cost, 60.0), (opp_cost + 15, 120.0), (opp_cost + 35, 80.0)], mtu)

    gmc = gas_price_at_hour(hour) / 0.38 + 75.0 * 0.18 + 4.0
    gas = supply_curve([(gmc, 100.0), (gmc + 10, 30.0), (gmc + 25, 20.0)], mtu)

    portfolio = merge_curves([wc, ror, res, gas])

    vols_wind.append(float(wc.total_volume))
    vols_ror.append(float(ror.total_volume))
    vols_res.append(float(res.total_volume))
    vols_gas.append(float(gas.total_volume))
    marginal_prices.append(float(portfolio.max_price))

print(f"Built {len(mtu_list)} portfolio curves")
peak_total = max(vols_wind[i]+vols_ror[i]+vols_res[i]+vols_gas[i] for i in range(96))
print(f"Peak portfolio volume: {peak_total:.0f} MW")

In [ ]:
hours = [i / 4 for i in range(96)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
fig.suptitle(
    "Fjord Energy Portfolio — Intraday Supply Profile (2026-03-15)",
    fontsize=13, fontweight="bold"
)

ax1.stackplot(
    hours, vols_wind, vols_ror, vols_res, vols_gas,
    labels=["Wind (NO2)", "RoR Hydro (NO5)", "Reservoir (NO1)", "Gas OCGT (NO2)"],
    colors=["#27ae60", "#2980b9", "#8e44ad", "#e74c3c"],
    alpha=0.82,
)
ax1.set_ylabel("Volume (MW)")
ax1.set_title("Portfolio Volume by Source")
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(0)

ax2.plot(hours, marginal_prices, color="#e74c3c", linewidth=1.8, label="Marginal price")
ax2.fill_between(hours, marginal_prices, alpha=0.1, color="#e74c3c")
ax2.set_ylabel("Marginal Price (EUR/MWh)")
ax2.set_xlabel("Hour (CET)")
ax2.set_title("Portfolio Marginal Price (most expensive MW offered)")
ax2.set_xticks(range(0, 25, 2))
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("portfolio_profile.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved: portfolio_profile.png")

## 6. Curve Transformations — Scenario Analysis

The library provides `scale_curve` and `clip_curve` for scenario and sensitivity analysis.

In [ ]:
# Scenario 1: Gas price shock — rebuild gas curve at +EUR 15/MWh higher price
gas_marginal_shocked = gas_marginal + 15.0
gas_curve_shocked = supply_curve([
    (gas_marginal_shocked, 100.0),
    (gas_marginal_shocked + 10.0, 30.0),
    (gas_marginal_shocked + 25.0, 20.0),
], ref_mtu)

# Scenario 2: Wind curtailment — scale volumes down by 20%
wind_curtailed = scale_curve(wind_curve, Decimal("0.8"))

# Scenario 3: Cap portfolio offers below EUR 100/MWh (regulatory price cap)
portfolio_capped = clip_curve(portfolio_curve, max_price=Decimal("100"))

print("Transformation results:")
print(f"  Gas curve (base):       {float(gas_curve.min_price):.1f}–{float(gas_curve.max_price):.1f} EUR/MWh")
print(f"  Gas curve (+15 shock):  {float(gas_curve_shocked.min_price):.1f}–{float(gas_curve_shocked.max_price):.1f} EUR/MWh")
print(f"  Wind original volume:   {float(wind_curve.total_volume):.0f} MW")
print(f"  Wind curtailed (80%):   {float(wind_curtailed.total_volume):.0f} MW")
print(f"  Portfolio (all steps):  {float(portfolio_curve.total_volume):.0f} MW")
print(f"  Portfolio (≤100 EUR):   {float(portfolio_capped.total_volume):.0f} MW")

# Show impact on portfolio aggregate under gas shock
portfolio_shocked = merge_curves([wind_curve, ror_hydro_curve, reservoir_hydro_curve, gas_curve_shocked])

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle("Portfolio Curve — Base vs Gas Price Shock (+€15/MWh)", fontsize=12, fontweight="bold")

x_base, y_base = curve_to_step_xy(portfolio_curve)
x_shock, y_shock = curve_to_step_xy(portfolio_shocked)

ax.step(x_base, y_base, where="post", color="#2980b9", linewidth=2, label="Base case")
ax.fill_between(x_base, y_base, alpha=0.12, color="#2980b9", step="post")
ax.step(x_shock, y_shock, where="post", color="#e74c3c", linewidth=2,
        linestyle="--", label="Gas shock (+€15/MWh)")

ax.set_xlabel("Cumulative Volume (MW)")
ax.set_ylabel("Price (EUR/MWh)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("scenario_analysis.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved: scenario_analysis.png")

## Summary

This notebook demonstrated portfolio-level merit order curve construction:

- **`from_dict_list`** constructs supply curves from plain Python data — ideal for integrating
  with asset management systems or model outputs
- **`merge_curves`** aggregates volumes at common price levels across all assets, producing
  the portfolio merit order curve in a single call
- **`scale_curve`** scales volumes for curtailment or derating scenarios
- **`clip_curve`** enforces price caps or offer boundaries
- The aggregate merit order directly informs bidding strategy: bids priced at each step's
  marginal cost maximise acceptance probability while meeting minimum revenue requirements

Next: see `04_order_book_and_validation.ipynb` for end-to-end order book management
and EUPHEMIA compliance validation.